# FallVision 04: training, model choice on validation, learning curves

Pipeline step 2 of 3. Everything is done by the scripts in `src/` through `run_pipeline.py`.

| Stage | Script | Output |
|---|---|---|
| train | `src/train_gru.py` | **final model**: GRU with augmentation and hard negatives x3 -> `models/gru_hard.pt`; comparison models: GRU without hard-negative weighting, GRU without augmentation (`src/train_gru.py --no-aug`), Random Forest (`src/train_rf.py`) |
| train | `src/plot_curves.py` | training / validation loss and macro-F1 per epoch -> `results/training_curves.png` |
| choose | `src/fuse.py` | every model / score / threshold compared on validation -> `results/fusion.md` |

**Before running (Kaggle)**
- Input (*Add Input > Your Work > Notebooks*): the output of `03_poses_features_windows` only.
- *Settings*: Accelerator **GPU T4 x1**, Internet **on**. *Add-ons > Secrets*: `GITHUB_TOKEN` on.
- Time: about 20-30 minutes. **Save Version > Save & Run All (Commit)**, or run the cells one by one.

**Next:** `05_evaluate` with this notebook's output as one of its inputs. The trained model is in
this notebook's Output at `FallVision/models/gru_hard.pt`.

In [ ]:
# ---- settings: the only cell you may need to change ----
REPO_URL = "https://github.com/Prish1509/project_fallvision.git"   # your repository (public)
BRANCH = "main"
PUSH_RESULTS = True             # push the result files back to the repository (needs the GITHUB_TOKEN secret)
REPO = "/kaggle/working/project_fallvision"   # where the code is cloned in this notebook
WORK = "/kaggle/working/work"         # pose caches, features, windows, probabilities (kept in the Output)

## 1. Code and packages

In [ ]:
import os, shutil, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
import kaggle_helpers as kh

# the exact versions from requirements.txt of the packages Kaggle does not have (torch is Kaggle's own)
pins = [l.split("#")[0].strip() for l in open("requirements.txt")
        if l.split("==")[0].strip() in ("ultralytics", "ultralytics-thop", "lap")]
kh.run([sys.executable, "-m", "pip", "install", "-q", *pins])

import torch
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - turn it on: Settings > Accelerator > GPU T4 x1")

## 2. Input: the windows made by notebook 03

In [ ]:
WIN = kh.find_dir_with("train.npz", "val.npz", "train_videos.csv", "val_videos.csv")
assert WIN, "windows not found: add the output of 03_poses_features_windows as an input"
print("windows:", WIN)
if not os.path.isdir(f"{WORK}/windows"):
    shutil.copytree(WIN, f"{WORK}/windows")
print(open(f"{WORK}/windows/summary.md").read())

## 3. Training (final model + comparison models) and learning curves

In [ ]:
kh.run([sys.executable, "run_pipeline.py", "--work", WORK, "--from", "train", "--to", "train", "--all-models"])

In [ ]:
from IPython.display import Image, display
display(Image("results/training_curves.png"))
print(open("results/training_curves.md").read())
kh.show_change(REPO, "results/gru_hard.md")

## 4. Model, score and threshold compared on validation

In [ ]:
kh.run([sys.executable, "run_pipeline.py", "--work", WORK, "--all-models", "--from", "choose", "--to", "choose"])
kh.show_change(REPO, "results/fusion.md")

## 5. Save the results to GitHub (the model itself stays in the Output: it is not put in git)

In [ ]:
if PUSH_RESULTS:
    kh.push(REPO, ["results"], "Pipeline 04: training, learning curves, model choice on validation", REPO_URL, BRANCH)
print("Trained model:", os.path.join(REPO, "models", "gru_hard.pt"),
      f"({os.path.getsize('models/gru_hard.pt') / 1e6:.1f} MB)")